# دفتر توليد صور البرمية القديمة

هذا دفتر Jupyter التنفيذي لمسار البيانات الصناعية. الكود المصدر للتوليد يبقى في `training/synthetic/generate_old_permic_synthetic.py`؛ تستدعي الخلايا أدناه الدوال الفعلية منه ولا تنسخ منطقًا بديلًا.

> ابدأ بـ S0، ثم أضف تغييرًا واحدًا فقط لكل مرحلة. لا تشغّل S1 أو S2 أو تدريب YOLO قبل فحص ناتج المرحلة السابقة وملفات manifest وassets.jsonl.

In [ ]:
# 1) تثبيت مسارات المشروع والخط. عدّل FONT_PATH فقط عند استخدام خط مرخّص مختلف.
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'training').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

OUTPUT_ROOT = PROJECT_ROOT / 'artifacts' / 'synthetic'
FONT_PATH = Path('/usr/share/fonts/truetype/noto/NotoSansOldPermic-Regular.ttf')
GENERATOR_PATH = PROJECT_ROOT / 'training' / 'synthetic' / 'generate_old_permic_synthetic.py'
VALIDATOR_PATH = PROJECT_ROOT / 'scripts' / 'validate_synthetic_dataset.py'

assert GENERATOR_PATH.is_file(), GENERATOR_PATH
assert FONT_PATH.is_file(), FONT_PATH
print('المشروع:', PROJECT_ROOT)
print('المولد:', GENERATOR_PATH)
print('الخط:', FONT_PATH)

## S0 · baseline نظيف ومتوازن

هذه المرحلة تنتج صورة واحدة ووسم YOLO واحد لكل حرف. التوازن موزع لكل فئة داخل train/val/test.

In [ ]:
# 2) استيراد واجهة Python الحقيقية للمولد، لا تنسخ منطق الرسم هنا.
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from training.synthetic.generate_old_permic_synthetic import PROFILES, write_dataset

S0_OUTPUT = OUTPUT_ROOT / 'S0-v1-unicode-clean'
manifest_s0 = write_dataset(
    output_dir=S0_OUTPUT,
    profile=PROFILES['unicode-clean'],
    samples=7600,
    seed=10350,
    image_size=640,
    font_size=58,
    font_path=FONT_PATH,
    layout='isolated-glyph',
    balanced_classes=True,
)
print(json.dumps(manifest_s0, ensure_ascii=False, indent=2))

## S0-d1 · تشويه مضبوط

غيّر هنا profile واحدًا فقط مع إبقاء حجم الصورة والخط والبذرة موثقين. هذا اختبار متانة للحرف ولا يحاكي مخطوطة تاريخية.

In [ ]:
# 3) تجربة تشويه أولى: المتغير المختلف الوحيد هو controlled-deformation.
S0D1_OUTPUT = OUTPUT_ROOT / 'S0-d1-controlled-deformation'
manifest_s0d1 = write_dataset(
    output_dir=S0D1_OUTPUT,
    profile=PROFILES['controlled-deformation'],
    samples=7600,
    seed=20350,
    image_size=640,
    font_size=58,
    font_path=FONT_PATH,
    layout='isolated-glyph',
    balanced_classes=True,
)
print(json.dumps(manifest_s0d1, ensure_ascii=False, indent=2))

## S1 · أسطر حروف منظمة

تنتج هذه المرحلة محارف مستقلة مرتبة بصريًا فقط. لا تحمل السلاسل المولدة دلالة كلمة أو معجم.

In [ ]:
# 4) لا تشغّل هذه الخلية إلا بعد قبول S0/S0-d1.
S1_OUTPUT = OUTPUT_ROOT / 'S1-v1-ordered-lines'
manifest_s1 = write_dataset(
    output_dir=S1_OUTPUT,
    profile=PROFILES['manuscript-inspired'],
    samples=1000,
    seed=30350,
    image_size=640,
    font_size=46,
    font_path=FONT_PATH,
    layout='ordered-lines',
)
print(json.dumps(manifest_s1, ensure_ascii=False, indent=2))

## S2 · صفحات صناعية منظمة

تضيف S2 مناطق وأعمدة وترتيب قراءة في `assets.jsonl`، مع بقاء جميع مربعات YOLO على مستوى الحرف.

In [ ]:
# 5) لا تشغّل هذه الخلية إلا بعد مراجعة S1.
S2_OUTPUT = OUTPUT_ROOT / 'S2-v1-structured-pages'
manifest_s2 = write_dataset(
    output_dir=S2_OUTPUT,
    profile=PROFILES['manuscript-inspired'],
    samples=600,
    seed=40350,
    image_size=640,
    font_size=38,
    font_path=FONT_PATH,
    layout='structured-pages',
)
print(json.dumps(manifest_s2, ensure_ascii=False, indent=2))

## التحقق قبل التدريب

لا تدخل أي حزمة إلى تدريب YOLO قبل اجتياز المدقق. سجّل البذرة ونسخة المولد ونتيجة التحقق في commit أو سجل التغيير.

In [ ]:
# 6) بدّل DATASET_TO_VALIDATE بالحزمة التي قبلتها فقط.
import subprocess

DATASET_TO_VALIDATE = S0_OUTPUT
subprocess.run([sys.executable, str(VALIDATOR_PATH), str(DATASET_TO_VALIDATE)], check=True)
print('اجتازت الحزمة الفحص. لا يعني ذلك بعد وجود وزن YOLO مدرّب أو أداء على مخطوطات حقيقية.')